# Objective
Genetic Programming is one of the evolutionary algorith that have human-readable rule. In this project we want to try to make an algorithm that can be readable by expert for Melanoma which is a skin cacner. In medical world they use **7-point Checklist Concept** (7P-CP) and **patient metadata**. This checklist contains "ground-truth" of melanoma detection. 

With GP we want to make create algorithm that can be use to see if there is any similarity in the rule making based compare to 7P-CP. In three settings:

1. **Oracle:** concept inputs are dermatologist annotations.
2. **MIP**: Baseline comparing with other algorithm. Recreating algorithm from Zhang, Morris, Ustun & Ghassemi (NeurIPS 2021), “Learning Optimal Predictive Checklists”
3. **Realistic:** concept inputs are scored from the image by a frozen medical VLM (MONET).

# Mixed-Integer Programming Baseline
MIP (Mixed-Integer Programming) is an optimization model that expresses a decision problem in mathematical form and passes it to a solver. It has three parts:
1. Decision variables: the choices to be made, which must be integers (0/1, e.g. include an item or not).
2. Constraints: the rules every solution must satisfy.
3. Objective function: the quantity to minimize or maximize (e.g. number of prediction errors).

Instead of enumerating every possible combination, the solver prunes combinations that cannot be optimal, so it can find the best solution and prove its optimality efficiently on problems of moderate size.

In [38]:
import time
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from docplex.mp.model import Model     # IBM CPLEX modelling API: pip install docplex cplex

ROOT = Path.cwd()
DATASETS = [ROOT / "release_v0" / "release_v0" / "meta", 
            ROOT / "data" / "raw" / "release_v0" / "meta"]
DATA = next((p for p in DATASETS if p.exists()), None)
assert DATA is not None, f"Derm7pt meta folder not found"
MONET_SCORES = ROOT / "data" / "processed" / "monet_concept_scores.csv"
MONET_PRESENT = 0.0
RESULTS = ROOT / "results"
TIME_LIMIT = 60.0 # limit for MIP solver in seconds
N_MAX = 8 # based on Zhang et al
SEED = 0

## Solver
Implementing paper solver based on "Learning Optimal Predictive
Checklists" by Zhang et al. 

Source code: https://github.com/MLforHealth/predictive_checklists

IBM CPLEX is being used for the paper through docplex (`pip install docplex cplex`)

- `cplex` package is the free community edition that limited to 1k variables and 1k constraint. This model has n+d+2 variables and n+2+(one eachgroup) constraint.
- `cplex` stops at 0.01 =% relative gap by default which can be indicated by OPTIMAL


In [39]:

@dataclass
class Checklist:
    '''
    This is where we store the result of screening. This part only stored MIP result and used it to predict
    new patient. 

    Output will be Optimal or Feasible
    Optimal: Means this is the best card
    Feasible: Reached time limit but not proven being optimal
    '''
    items: list # names of the chosen items
    idx: list # column indices in X
    M: int # threshold of checklist if >=M ==> Positive
    gap: float # Optimal gap from the best solution 0% means good
    status: str # Optimal or Feasible result

    def predict(self, X):
        X = np.asarray(X)
        return (X[:, self.idx].sum(axis=1) >= self.M).astype(int) # Calculated mark from each patient and check if reach threshold

    def __str__(self):
        '''
        Print patient checklist
        '''
        rows = [f"Predict POSITIVE if {self.M}+ of these {len(self.items)} items are checked:"]
        rows += [f"  [ ] {name}" for name in self.items]
        rows.append(f"  (optimality gap {self.gap:.1%}, {self.status})")
        return "\n".join(rows)

In [ ]:
def fit_checklist(X, y, names=None, groups=None, n_max=8, force_M=None,
                  class_weight=None, max_fnr=None, max_fpr=None,
                  time_limit=60.0, workers=8, seed=0):
    '''
    Trying all combination and writting the problem into mathematic equation then parse to CPLEX.

    Args:
    X               n x d binary item matrix 0/1
    y               n labels
    name            item names
    groups          length-d group id per item at least one item per group is being used as in paper
    n_max           maximum number of items N
    force_M         Threshold number
    class_weight    None => minimize number of mistakes
                    'balanced' => minimize FNR + FPR 
    '''

    # Prepare datasets
    X = (np.asarray(X) > 0).astype(int)
    y = np.asarray(y).astype(int)
    n, d = X.shape # row = paitnet; column = item (blue-whitish veil etc)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0) #pos=melanoma; neg=non-melanoma
    n_max = min(n_max, d)
    names = list(names) if names is not None else [f"x{j}" for j in range(d)]

    mdl = Model(name="checklist")
    lam = mdl.binary_var_list(d, name="lam") # marker for item j in the checklist
    z = mdl.binary_var_list(n, name="z") # 1 wrong checklist for patient i vice versa 
    N = mdl.integer_var(lb=1, ub=n_max, name="N") # N item  in checklist with min 1 item
    M = mdl.integer_var(lb=1, ub=n_max, name="M") # Same like N but for threshold

    def score(i):
        return mdl.sum(lam[j] for j in np.flatnonzero(X[i]))


    mdl.add_constraint(N == mdl.sum(lam)) # N and sum from checklist consistent
    mdl.add_constraint(M <= N) #Compare with threshold
    # Loop to check whether the checklist is true or not. Thus wer know the number mistakes from the checklist
    for i in pos:  #Positive patients                                   
        mdl.add_constraint(n_max * z[i] >= M - score(i))
    for i in neg:
        B = min(int(X[i].sum()), n_max)
        if B > 0:
            mdl.add_constraint(B * z[i] >= score(i) - M + 1)
    l_pos = mdl.sum(z[i] for i in pos) # n mistakes on positive
    l_neg = mdl.sum(z[i] for i in neg) # vice versa

    if groups is not None:                                  # at most one item per group
        for g in set(groups):
            members = [j for j in range(d) if groups[j] == g]
            if len(members) > 1:
                mdl.add_constraint(mdl.sum(lam[j] for j in members) <= 1)
    # Another threshold
    if force_M is not None:
        mdl.add_constraint(M == force_M) # At least 1 checklist
    if max_fnr is not None:
        mdl.add_constraint(l_pos <= int(np.floor(max_fnr * len(pos)))) # Melanoma threhold
    if max_fpr is not None:
        mdl.add_constraint(l_neg <= int(np.floor(max_fpr * len(neg)))) # Threshold for 

    w_pos, w_neg = (len(neg), len(pos)) if class_weight == "balanced" else (1, 1)
    # This part we want to inimize the mistakes of the solver by considering how many FN skipped N items and weight
    K_N = n_max + 1 # weight of N larger than any M
    K_E = K_N * n_max + n_max + 1 # Weight of mistakes larger than any K_N*N + M mistakes outweights change in items
    mdl.minimize(K_E * (w_pos * l_pos + w_neg * l_neg) + K_N * N + M) 

    mdl.parameters.timelimit.set(float(time_limit)) #60 seconds
    mdl.parameters.threads.set(workers)
    mdl.parameters.randomseed.set(seed)
    mdl.parameters.mip.tolerances.mipgap.set(1e-4)  # CPLEX stops at a 0.01% gap by default
    mdl.parameters.mip.tolerances.absmipgap.set(0.0) # 0 makes OPTIMAL mean certified optimal
    #run solver
    sol = mdl.solve()
    if sol is None:
        raise RuntimeError(f"No checklist found: {mdl.solve_details.status}")

    # Objective is in integer so value is rounded up
    v_max = round(sol.objective_value)
    v_min = np.ceil(mdl.solve_details.best_bound - 1e-6)
    gap = 0.0 if v_max == 0 else max(0.0, 1.0 - v_min / v_max)
    idx = [j for j in range(d) if sol.get_value(lam[j]) > 0.5]
    M_val = int(round(sol.get_value(M)))
    mdl.end()
    return Checklist([names[j] for j in idx], idx, M_val, gap, "OPTIMAL" if gap == 0 else "FEASIBLE")




In [41]:
class AdaptiveBinarizer:
    '''
    Checklist card only consist binary number 
    thus making it adaptive with continous variables are needed by changing into binary  
    '''

    def __init__(self, categorical=(), binary=()):
        self.categorical, self.binary = categorical, binary

    def fit(self, df):
        self.levels_ = {c: sorted(df[c].dropna().unique()) for c in self.categorical}
        return self

    def transform(self, df):
        cols, names, groups = [], [], []
        for c in self.binary:
            cols.append((df[c].to_numpy() > 0).astype(int)); names.append(c); groups.append(c)
        for c, levels in self.levels_.items():
            for v in levels:
                cols.append((df[c].to_numpy() == v).astype(int))
                names.append(f"{c} = {v}"); groups.append(c)
        return np.column_stack(cols), names, groups

## Load Data

In [42]:
meta = pd.read_csv(DATA / "meta.csv")
split_idx = {s: pd.read_csv(DATA / f"{s}_indexes.csv")["indexes"].to_numpy()
             for s in ["train", "valid", "test"]}
meta["split"] = ""
for s, ix in split_idx.items():
    meta.loc[ix, "split"] = s

meta["is_mel"] = meta["diagnosis"].str.startswith("melanoma").astype(int)
print(meta.groupby("split").agg(cases=("is_mel", "size"), melanomas=("is_mel", "sum")))

       cases  melanomas
split                  
test     395        101
train    413         90
valid    203         61


## Oracle concepts and metadata mapping
Oracle means findings from dermatologist expert. 
Maps the meta.csv to 7P-CP major indications 2pts and minor 1 pt

| Criterion | Points | Positive when |
|---|---|---|
| pigment_network | 2 | `atypical` |
| blue_whitish_veil | 2 | `present` |
| vascular_structures | 2 | `linear irregular` or `dotted` (other types, incl. `within regression`, are not atypical) |
| streaks | 1 | `irregular` |
| dots_and_globules | 1 | `irregular` |
| pigmentation | 1 | `diffuse irregular` or `localized irregular` |
| regression_structures | 1 | anything but `absent` |

Metadata used: `sex`, `location` , `elevation`. 
Excluded : `management`, `level_of_diagnostic_difficulty` (possible leakage)

In [ ]:
CONCEPTS = {  # name and points
    "atypical network":        (2, meta["pigment_network"] == "atypical"),
    "blue-whitish veil":       (2, meta["blue_whitish_veil"] == "present"),
    "atypical vascular":       (2, meta["vascular_structures"].isin(["linear irregular", "dotted"])),
    "irregular streaks":       (1, meta["streaks"] == "irregular"),
    "irregular dots/globules": (1, meta["dots_and_globules"] == "irregular"),
    "irregular pigmentation":  (1, meta["pigmentation"].str.contains("irregular")),
    "regression structures":   (1, meta["regression_structures"] != "absent"),
}
for name, (_, s) in CONCEPTS.items():
    meta[name] = s.astype(int)
concepts = list(CONCEPTS)
points = pd.Series({k: v[0] for k, v in CONCEPTS.items()})
meta["pcp7_score"] = meta[concepts].mul(points).sum(axis=1)

META_COLS = ["sex", "location", "elevation"]
meta[concepts + META_COLS + ["split", "is_mel"]].head()

,atypical network,blue-whitish veil,atypical vascular,irregular streaks,irregular dots/globules,irregular pigmentation,regression structures,sex,location,elevation,split,is_mel
0,0,0,0,0,0,0,0,female,abdomen,nodular,train,0
1,0,0,0,0,1,0,0,female,head neck,palpable,valid,0
2,0,0,0,0,1,0,0,female,head neck,palpable,train,0
3,0,1,0,0,1,0,1,male,lower limbs,palpable,test,0
4,0,0,0,0,0,1,0,female,upper limbs,palpable,train,0


## Oracle Table

All patient data in one table and split into three parts, every method is trained and tested on exactly the same patients.

- Task: Deciding melanoma or from 1011 skin lesions using official split Derm7pt
| Part | Patients | Used for |
|---|---|---|
| train | 413 | build the checklist |
| valid | 203 | choose the best checklist length |
| test | 395 | final score, looked at once |

This table will be used for MIP, 7P-CP and GP. 

In [ ]:
inputs = {"oracle": meta[["case_num"] + concepts + META_COLS + ["split", "is_mel", "pcp7_score"]]}

settings = {s: {p: df[df["split"] == p].reset_index(drop=True) for p in ["train", "valid", "test"]}
            for s, df in inputs.items()}
for s, parts in settings.items():
    print(s, {p: (len(d), int(d["is_mel"].sum())) for p, d in parts.items()})

oracle {'train': (413, 90), 'valid': (203, 61), 'test': (395, 101)}


## Metrics and fitting helpers
This section define how we score the rule. Containing helper functions that fit a checklist and evaluate

**How we score a rule** (from the project concept):

| Metric | Plain meaning |
|---|---|
| Sensitivity | Of all melanomas, how many did the rule catch? Missing one is costly |
| Specificity | Of all non-melanomas, how many did the rule correctly clear? |
| Balanced accuracy | The average of the two, so the rarer melanoma class counts as much as the common one |
| Rule size | How long the rule is. Shorter is easier for a clinician to read |

MIP is trained with `class_weight="balanced"` thus aims for high balanced accuracy

Helper functions:
- `rates` computes sensitivity, specificity and balanced accuracy.
- `rule_size` counts the nodes of a checklist.
- `feature_sets` defines the two inputs we compare: concepts only, and concepts + metadata.
- `fit_one` fits one checklist on the train part.
- `evaluate` scores a fitted checklist on any part (train, valid or test).

In [45]:
def rates(y, p):
    '''
    args:
    sens        from all melanoma patients how many percents succesffully mark
    spec        from all healthy patients how many percent not reffered to doctor
    bal_acc     averge from both metrics
    '''
    y, p = np.asarray(y), np.asarray(p)
    sens = ((p == 1) & (y == 1)).sum() / (y == 1).sum()
    spec = ((p == 0) & (y == 0)).sum() / (y == 0).sum()
    return dict(sens=sens, spec=spec, bal_acc=(sens + spec) / 2)

VARIANTS = {"MIP": dict()} 

def feature_sets():
    return {"concepts":          {"binary": concepts},
            "concepts+metadata": {"binary": concepts, "categorical": META_COLS}}

def fit_one(parts, fset_args, variant, n_max):
    tr = parts["train"]
    binz = AdaptiveBinarizer(**fset_args).fit(tr)                   # levels from train only
    X, names, groups = binz.transform(tr)
    cl = fit_checklist(X, tr["is_mel"].to_numpy(), names=names, groups=groups, n_max=n_max,
                       class_weight="balanced", time_limit=TIME_LIMIT, seed=SEED, **VARIANTS[variant])
    return binz, cl

def evaluate(parts, binz, cl, split):
    X, _, _ = binz.transform(parts[split])
    return rates(parts[split]["is_mel"], cl.predict(X))

## 7P-CP scoring checklist

In [ ]:
def pcp7_predict(d):
    return (d["pcp7_score"] >= 3).astype(int).to_numpy()

rows = []
for s, parts in settings.items():
    for p in ["train", "valid"]:
        rows.append(dict(setting=s, split=p, **rates(parts[p]["is_mel"], pcp7_predict(parts[p]))))
pd.DataFrame(rows).round(3)

,setting,split,sens,spec,bal_acc
0,oracle,train,0.933,0.811,0.872
1,oracle,valid,0.902,0.859,0.880


Human 7P-CP scoring resulting 93% accuraacy with 19% falsed reffered to doctor (healthy patient)

## Fit MIP and validation

Each features set solve for n_max in 1.8 on train. Validation picks n_max with highest `balanced_accuracy` 

In [47]:
parts = settings["oracle"]

fits, log = {}, []
for fset, fset_args in feature_sets().items():
    for n_max in range(1, N_MAX + 1):
        binz, cl = fit_one(parts, fset_args, "MIP", n_max)
        tr, va = evaluate(parts, binz, cl, "train"), evaluate(parts, binz, cl, "valid")
        fits[(fset, n_max)] = (binz, cl)
        log.append(dict(features=fset, n_max=n_max, N=len(cl.items), M=cl.M,
                        train_bal_acc=tr["bal_acc"], val_bal_acc=va["bal_acc"], val_sens=va["sens"],
                        gap=cl.gap, status=cl.status))
log = pd.DataFrame(log)
print("all optimal:", bool((log["status"] == "OPTIMAL").all()))

all optimal: False


In [48]:
best = (log.sort_values(["val_bal_acc", "N", "M"], ascending=[False, True, True])
           .groupby("features").head(1)
           .sort_values("features").reset_index(drop=True))
best.round(3)

,features,n_max,N,M,train_bal_acc,val_bal_acc,val_sens,gap,status
0,concepts,4,4,2,0.857,0.868,0.869,0.0,OPTIMAL
1,concepts+metadata,5,5,2,0.860,0.873,0.902,0.0,OPTIMAL


Features
Concept: choosing 4 item with rule more than 2 then refer doctor
Concetp+Metadata: Choosing 5 items with rule more than 2 then refer to doctor

Including metadata only increasing validation 0.01 (val_bal_acc)

## Final evaluation on test

The 7P-CL and the selected MIP checklist per (setting, features) are scored on test. For the human
pcp7, `N` is its 7 criteria and `M` is its point threshold.

In [ ]:
rows, preds = [], []

def add(features, method, N, M, pred, gap=np.nan, status="-"):

    tr, va, te = (rates(parts[k]["is_mel"], pred[k]) for k in ["train", "valid", "test"])
    rows.append(dict(features=features, method=method, N=N, M=M,
                     train_bal_acc=tr["bal_acc"], val_bal_acc=va["bal_acc"], test_bal_acc=te["bal_acc"],
                     test_sens=te["sens"], test_spec=te["spec"], gap=gap, status=status))
    preds.append(pd.DataFrame(dict(case_num=parts["test"]["case_num"], features=features,
                                   method=method, is_mel=parts["test"]["is_mel"], pred=pred["test"])))

add("concepts", "human 7P-CL", 7, 3, {k: pcp7_predict(d) for k, d in parts.items()})
for _, r in best.iterrows():
    binz, cl = fits[(r["features"], int(r["n_max"]))]
    add(r["features"], "MIP", r["N"], r["M"], {k: cl.predict(binz.transform(d)[0]) for k, d in parts.items()},
        gap=r["gap"], status=r["status"])

results = pd.DataFrame(rows)

# save into csv for future comparison
RESULTS.mkdir(exist_ok=True)
results.to_csv(RESULTS / "mip_baseline_test.csv", index=False)
pd.concat(preds).to_csv(RESULTS / "mip_baseline_test_predictions.csv", index=False)
results.round(3)

,features,method,N,M,train_bal_acc,val_bal_acc,test_bal_acc,test_sens,test_spec,gap,status
0,concepts,human 7P-CL,7,3,0.872,0.880,0.878,0.941,0.816,NaN,-
1,concepts,MIP,4,2,0.857,0.868,0.847,0.871,0.823,0.0,OPTIMAL
2,concepts+metadata,MIP,5,2,0.860,0.873,0.844,0.871,0.816,0.0,OPTIMAL


From this result we can see that manual checklist by doctor still perfom best during the test showing that MIP solver cannot beat the human intervention due to the difference in giving weights of each items that got check list


In [ ]:
for _, r in best.iterrows():
    print("---", r["features"])
    print(fits[(r["features"], int(r["n_max"]))][1], "\n")
    #printing cheklist result from MIP

--- concepts
Predict POSITIVE if 2+ of these 4 items are checked:
  [ ] atypical network
  [ ] blue-whitish veil
  [ ] irregular streaks
  [ ] irregular dots/globules
  (optimality gap 0.0%, OPTIMAL) 

--- concepts+metadata
Predict POSITIVE if 2+ of these 5 items are checked:
  [ ] atypical network
  [ ] blue-whitish veil
  [ ] irregular streaks
  [ ] irregular dots/globules
  [ ] elevation = nodular
  (optimality gap 0.0%, OPTIMAL) 

